## Generación de la Vector Store

In [ ]:
from src.utils import cargar_datos

# Importar datos
df = cargar_datos()

In [ ]:
from src.rag import generate_vector_database

# Creación de la base de datos
generate_vector_database(df)

## Uso de RAG en pruebas

In [ ]:
from src.rag import obtener_retriever
from src.config import PATH_VECTOR_DB


# Generar retriever
retriever = obtener_retriever(k = 5 , path=PATH_VECTOR_DB)

In [ ]:
# Probar con un ejemplo
titulo = df.sample(1).title.iloc[0]
print(f"Buscando contexto para el título: '{titulo}'\n")

docs_recuperados = retriever.invoke(titulo)

for i, doc in enumerate(docs_recuperados):
    print(f"--- Documento Recuperado #{i+1} ---")
    print(f"ID Original de la fila: {doc.metadata.get('row_id')}")
    print(f"Título del registro indexado: {doc.metadata.get('original_title')} {doc.metadata.get('category')}")
    print(f"Descripción recuperada (Contexto):\n{doc.page_content}\n")

In [ ]:
from src.classifiers import predecir_categoria_rag, predecir_categoria

# Comparación de respuesta de LLM con RAG y sin RAG
titulo = "Lebron James will be president"

# FallBack en false para asegurar que el prompting es lo suficientemente bueno para que conteste como debe
print(f"Versión RAG:    {predecir_categoria_rag(titulo,retriever, fallback=False)}")
print(f"Versión no RAG: {predecir_categoria(titulo, fallback=False)}")

## Comparativas de los métodos de predicción (LLM - LLM + RAG - KNN + RAG)

In [ ]:
df_test = cargar_datos(conjunto="test")
df_test

In [ ]:
from tqdm import tqdm

from src.rag import obtener_retriever, generate_vector_database
from src.classifiers import predecir_categoria, predecir_categoria_rag, predecir_categoria_knn
from src.utils import fallback_postprocessing
from src.config import CLASS_TO_LABEL

In [ ]:
k = 5
espacios_distancia = ["cosine", "l2", "ip"]
retrievers = {}

# Crear los distintos retrievers
for space in espacios_distancia:
    print(f" ------ Configurando Vector Store para: {space} ------")
    
    vector_store = generate_vector_database(df, space=space, save=False)
    retriever = obtener_retriever(k, vector_store)
    
    retrievers[space] = retriever

print("Retrievers generados")
print(list(retrievers.keys()))

#### LLM

In [ ]:
y_true_no_rag = []
y_pred_no_rag = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando LLM sin RAG"):
    titulo = row["title"]
    label_real = row["label"]
    
    # Predicción
    pred_str = predecir_categoria(titulo, fallback=False)
    pred_str_fb = fallback_postprocessing(pred_str)
    
    # Conversión a ID (si no existe en las clases válidas, devuelve -1)
    pred_id = CLASS_TO_LABEL.get(pred_str_fb, -1)
    
    y_true_no_rag.append(label_real)
    y_pred_no_rag.append(pred_id)

#### LLM + RAG

In [ ]:
y_true_rag = []
y_pred_rag = {"cosine": [], "l2": [], "ip": []}

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando LLM con RAG"):
    titulo = row["title"]
    label_real = row["label"]
    
    y_true_rag.append(label_real)
    
    for dist, retriever in retrievers.items():
        pred_str = predecir_categoria_rag(titulo, retriever, fallback=False)
        pred_str_fb = fallback_postprocessing(pred_str)
        
        pred_id = CLASS_TO_LABEL.get(pred_str_fb, -1)
        y_pred_rag[dist].append(pred_id)

#### KNN

In [ ]:
y_true_knn = []
y_pred_knn = {"cosine": [], "l2": [], "ip": []}

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando KNN"):
    titulo = row["title"]
    label_real = row["label"]
    
    y_true_knn.append(label_real)
    
    for dist, retriever in retrievers.items():
        pred_str = predecir_categoria_knn(titulo, retriever)
        
        # Innecesario el -1, pero nunca sobra seguridad
        pred_id = CLASS_TO_LABEL.get(pred_str, -1)
        y_pred_knn[dist].append(pred_id)

#### Comparaciones

In [ ]:
import pandas as pd

df_analysis = pd.DataFrame({
    "y_true": y_true_rag,
    "llm_no_rag": y_pred_no_rag,
    "llm_rag_cosine": y_pred_rag["cosine"],
    "llm_rag_l2": y_pred_rag["l2"],
    "llm_rag_ip": y_pred_rag["ip"],
    "knn_cosine": y_pred_knn["cosine"],
    "knn_l2": y_pred_knn["l2"],
    "knn_ip": y_pred_knn["ip"]
})

total_muestras = len(df_analysis)

In [ ]:
# =======================================
# ANÁLISIS DE PREDICCIONES INVÁLIDAS (-1)
# =======================================
print("--- PREDICCIONES INVÁLIDAS (El modelo falló al dar una clase válida) ---")
invalids = (df_analysis == -1).sum()

print(f"LLM sin RAG:      {invalids['llm_no_rag']}")
print(f"LLM RAG (Cosine): {invalids['llm_rag_cosine']}")
print(f"LLM RAG (L2):     {invalids['llm_rag_l2']}")
print(f"LLM RAG (IP):     {invalids['llm_rag_ip']}")
print(f"KNN (Cosine):     {invalids['knn_cosine']}")
print(f"KNN (L2):         {invalids['knn_l2']}")
print(f"KNN (IP):         {invalids['knn_ip']}")

In [ ]:

# ==============================================================
# DISCREPANCIAS INTRA-MODELO (Efecto de la métrica de distancia)
# ==============================================================
print("--- DISCREPANCIAS INTRA-MODELO (Mismo modelo, distinta distancia) ---")

# Dentro del LLM con RAG
diff_rag_cos_l2 = (df_analysis["llm_rag_cosine"] != df_analysis["llm_rag_l2"]).sum()
diff_rag_cos_ip = (df_analysis["llm_rag_cosine"] != df_analysis["llm_rag_ip"]).sum()
diff_rag_l2_ip  = (df_analysis["llm_rag_l2"]     != df_analysis["llm_rag_ip"]).sum()

# Dentro del KNN
diff_knn_cos_l2 = (df_analysis["knn_cosine"] != df_analysis["knn_l2"]).sum()
diff_knn_cos_ip = (df_analysis["knn_cosine"] != df_analysis["knn_ip"]).sum()
diff_knn_l2_ip =  (df_analysis["knn_l2"]     != df_analysis["knn_ip"]).sum()

print("En LLM con RAG:")
print(f"  Cosine vs L2: {diff_rag_cos_l2} diferencias ({(diff_rag_cos_l2/total_muestras)*100:.2f}%)")
print(f"  Cosine vs IP: {diff_rag_cos_ip} diferencias ({(diff_rag_cos_ip/total_muestras)*100:.2f}%)")
print(f"  L2 vs IP:     {diff_rag_l2_ip } diferencias ({(diff_rag_l2_ip /total_muestras)*100:.2f}%)")


print("\nEn KNN:")
print(f"  Cosine vs L2: {diff_knn_cos_l2} diferencias ({(diff_knn_cos_l2/total_muestras)*100:.2f}%)")
print(f"  Cosine vs IP: {diff_knn_cos_ip} diferencias ({(diff_knn_cos_ip/total_muestras)*100:.2f}%)")
print(f"  L2 vs IP:     {diff_knn_l2_ip } diferencias ({(diff_knn_l2_ip /total_muestras)*100:.2f}%)")

print()
print()

# ===============================================
# DISCREPANCIAS INTER-MODELO (LLM vs LLM con RAG)
# ===============================================
print("--- DISCREPANCIAS INTER-MODELO (LLM vs LLM+RAG) ---")

diff_norag_rag_cos = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_cosine"]).sum()
diff_norag_rag_l2  = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_l2"]).sum()
diff_norag_rag_ip  = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_ip"]).sum()

print(f"LLM vs LLM+RAG (Cosine): {diff_norag_rag_cos} diferencias ({(diff_norag_rag_cos/total_muestras)*100:.2f}%)")
print(f"LLM vs LLM+RAG (L2):     {diff_norag_rag_l2} diferencias ({(diff_norag_rag_l2/total_muestras)*100:.2f}%)")
print(f"LLM vs LLM+RAG (IP):     {diff_norag_rag_ip} diferencias ({(diff_norag_rag_ip/total_muestras)*100:.2f}%)")

In [ ]:

# ===============================
# ANÁLISIS DE ACIERTOS (Accuracy)
# ===============================
print("--- ACIERTOS Y PRECISIÓN GLOBAL ---")

columnas_modelos = [
    "llm_no_rag", 
    "llm_rag_cosine", "llm_rag_l2", "llm_rag_ip",
    "knn_cosine", "knn_l2", "knn_ip"
]

for modelo in columnas_modelos:
    aciertos = (df_analysis[modelo] == df_analysis["y_true"]).sum()
    accuracy = (aciertos / total_muestras) * 100
    
    print(f"{modelo.ljust(15)}: {aciertos}/{total_muestras} aciertos ({accuracy:.2f}%)")


In [ ]:
casos_mejorados = df_analysis[
    (df_analysis["llm_no_rag"] != df_analysis["y_true"]) & 
    (df_analysis["llm_rag_cosine"] == df_analysis["y_true"])
]
print(f"El RAG arregló {len(casos_mejorados)} errores del LLM puro.")